In [1]:
import sys 

print(sys.executable)
print(sys.version)

c:\Projects\FlightRisk\.venv\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


phase 2,here i will continue localy and work on several point including :
local reproducibility,
validate raw data,
define modeling population,
prepare leakage-safe X and y

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn 
import kagglehub 
from pathlib import Path

print ("numpy version : ", np.__version__)
print ("pandas version : ", pd.__version__)
print ("matplotlib version : ", plt.matplotlib.__version__)
print ("sklearn version : ", sklearn.__version__)
print ("kagglehub version : ", kagglehub.__version__)
Path.cwd()

numpy version :  2.5.3
pandas version :  3.0.5
matplotlib version :  3.11.1
sklearn version :  1.9.0
kagglehub version :  1.0.2


c:\Projects\FlightRisk\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WindowsPath('c:/Projects/FlightRisk/notebooks')

In [3]:
sample_path = Path("../data/raw/flight_data_2024_sample.csv")
data_dictionary_path = Path("../data/raw/flight_data_2024_data_dictionary.csv")

sample_path.exists()
data_dictionary_path.exists()

flights = pd.read_csv(sample_path)
flights.shape
flights.head()

,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,...,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,4,18,4,2024-04-18,MQ,3535.0,DFW,"Dallas/Fort Worth, TX",Texas,...,0,151.0,144.0,119.0,835.0,0,0,0,0,0
1,2024,1,1,1,2024-01-01,AA,148.0,CLT,"Charlotte, NC",North Carolina,...,0,286.0,273.0,253.0,1773.0,0,0,0,0,0
2,2024,12,12,4,2024-12-12,9E,5440.0,CHA,"Chattanooga, TN",Tennessee,...,0,59.0,50.0,29.0,106.0,0,0,0,0,0
3,2024,4,8,1,2024-04-08,WN,1971.0,OMA,"Omaha, NE",Nebraska,...,0,180.0,177.0,163.0,1099.0,0,0,0,0,0
4,2024,2,16,5,2024-02-16,WN,862.0,BWI,"Baltimore, MD",Maryland,...,0,90.0,96.0,76.0,399.0,0,0,0,0,0


phase 1


In [4]:
#create a copy of mine data frame to work with
model_data = flights.copy()
print(model_data.shape)
#filtring the dataframe from cancelled and diverted flights
model_data = model_data[
    (model_data["cancelled"] == 0) 
    & (model_data["diverted"] == 0)  
    & (model_data["arr_delay"].notna())
    ]
print(model_data.shape)
#using assert to confirm that canclled and diverted flights are exluded from the dataframe
assert (model_data["cancelled"] == 0).all() ,"canclled values must be 0 for this dataset"
assert (model_data["diverted"] == 0).all() ,"diverted values must be 0 for this dataset"
assert (model_data["arr_delay"].notna()).all() , "NaN arr_delay values must be exluded from dataframe"

(10000, 35)
(9836, 35)


inspecting duplicates shemas and identifiers 

In [5]:
#inspecting duplicates and unique combinations identifiers 
print(model_data.duplicated().value_counts())
print(model_data["op_carrier_fl_num"].nunique())
candidate_key = model_data.duplicated(subset=["op_carrier_fl_num", "fl_date" , "op_unique_carrier", "origin"])
legs = model_data[model_data.duplicated(subset=["op_carrier_fl_num", "fl_date" , "op_unique_carrier"], keep=False)]
print(legs[["op_carrier_fl_num", "fl_date" , "op_unique_carrier", "origin", "dest", "crs_dep_time"]])

assert candidate_key.sum() == 0, "this unique candidate key combinition must produce no duplicates"

False    9836
Name: count, dtype: int64
4562
      op_carrier_fl_num     fl_date op_unique_carrier origin dest  \
400              5038.0  2024-01-03                9E    DTW  SDF   
597              1922.0  2024-08-05                AA    PIT  ORD   
1731              369.0  2024-07-21                WN    BWI  MCO   
2243             2683.0  2024-02-22                UA    PIT  ORD   
2490             5038.0  2024-01-03                9E    SDF  DTW   
3364              369.0  2024-07-21                WN    DEN  RIC   
3813             2683.0  2024-02-22                UA    ORD  SAN   
4239             5463.0  2024-05-26                OH    RIC  CLT   
4436             5463.0  2024-05-26                OH    CLT  RIC   
6386             1153.0  2024-09-06                DL    ATL  RSW   
6943             1922.0  2024-08-05                AA    ORD  PIT   
7061             3453.0  2024-09-23                MQ    DFW  FSM   
7458             3453.0  2024-09-23                MQ    F

Creating a three sepereate variables list including potenial features provided before departure ,
model target (arr_delay),
leakage variables which is the data that the model will not have during prediction ,such as actuall_dep_time 

In [6]:
#creating two lists holiding the columns for X candidates and leakage coloumns

columns_X = ["fl_date","op_unique_carrier","origin","dest","crs_dep_time"
                           ,"crs_arr_time","crs_elapsed_time","distance"]
leakage_columns = [
    "dep_time", "dep_delay", "taxi_out", "wheels_off", "wheels_on",
    "taxi_in", "arr_time", "actual_elapsed_time", "air_time",
    "carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"
]
redundant_exluded_columns = ["year","month","day_of_month","day_of_week","op_carrier_fl_num"
                             , "origin_city_name", "origin_state_nm","dest_city_name"
                             ,"dest_state_nm", "cancelled","cancellation_code", "diverted"]

target_column = "arr_delay"

#using the list to create new data frame for X and Y and leakage columns 

model_data_X = model_data[columns_X].copy()
model_data_leakage = model_data[leakage_columns]
model_redundant_exluded = model_data[redundant_exluded_columns]
model_data_y = model_data[target_column]

In [7]:
#working on refrence dataframe audit assertions

overlap_leakage = set(columns_X) & set(leakage_columns)
overlap_redundant = set(columns_X) & set(redundant_exluded_columns)
assert len(overlap_leakage) == 0, f"an overlap of {overlap_leakage} was found."
assert len(overlap_redundant) == 0, f"an overlap of {overlap_redundant} was found."
assert target_column not in model_data_X, f"target column should not be inside features X."
assert target_column not in model_data_leakage, f"target columns should not be in leakage columns"
data_columns_check = len(columns_X) + len(leakage_columns) + len(redundant_exluded_columns) + 1
print(model_data.shape, model_data_X.shape,model_data_y.shape)
assert data_columns_check == len(model_data.columns) ,f"the sum of all columns variables should match the original dataframe, a 'column has not been classified"

#creating a new y_clf for classification target and run an audit assert on it 

target_column_clf = "clf_arr_delay"
model_data_y_clf = (model_data_y >= 15).astype(int)
model_data_y_clf.name = target_column_clf
assert target_column_clf not in model_data_X, f"target column_clf should not be inside features X."
assert target_column_clf not in model_data_leakage, f"target column_clf should not be in leakage columns"
print(model_data_y_clf.value_counts(), model_data_y_clf.mean())


(9836, 35) (9836, 8) (9836,)
clf_arr_delay
0    7717
1    2119
Name: count, dtype: int64 0.2154331028873526


doing feature engineering on some candidate features 

In [8]:
#turning fl_date from str into datetime dtype using .to_datetime

model_data_X["fl_date"] = pd.to_datetime(model_data_X["fl_date"])
model_data_X["month"] = model_data_X["fl_date"].dt.month
model_data_X["day_of_week"] = model_data_X["fl_date"].dt.dayofweek
#print(model_data_X["month"].head(), model_data_X["dayofweek"].head())
#creating a new required dep_hour serie from crs_dep_hour also adding integrity auditing using assert
model_data_X["dep_hour"] = model_data_X["crs_dep_time"] // 100
assert model_data_X["dep_hour"].min() >= 0 and model_data_X["dep_hour"].max() <=23 ,"a valid dep hour must be between 0 and 23"
dep_minutes = model_data_X["crs_dep_time"] % 100
assert dep_minutes.min() >= 0 and dep_minutes.max() <= 59, "a valid dep minutes must be between 0 and 59"
print(model_data.shape)
print(model_data_X.shape)

(9836, 35)
(9836, 11)


working on chronological train/validate/test split of data

In [9]:
print(f"Earliest date : {model_data_X["fl_date"].min()}, Latest date {model_data_X["fl_date"].max()}")

month_flights_inspect = model_data_X.groupby("month").agg({"month" : "count"})
print(month_flights_inspect)

#applying chronological splting mask to use later for slicing the dataframe

train_mask =  model_data_X["fl_date"] < "2024-09-01"
validate_mask = model_data_X["fl_date"].between("2024-09-01", "2024-10-31")
test_mask = model_data_X["fl_date"].between("2024-11-01", "2024-12-31")
#apply the mask filter for slicing the training (X) and target data (y)
model_train_X = model_data_X[train_mask]
model_train_y =  model_data_y[train_mask]
model_train_yclf = model_data_y_clf[train_mask]
#use of the validate mask to slice the validate dataset
model_val_X = model_data_X[validate_mask]
model_val_y = model_data_y[validate_mask]
model_val_yclf = model_data_y_clf[validate_mask]
#using the test mask to slice the test dataset
model_test_X = model_data_X[test_mask]
model_test_y = model_data_y[test_mask]
model_test_yclf = model_data_y_clf[test_mask]
print( f"splited data for model training X : {model_train_X.shape},y : {model_train_y.shape}, y_clf : {model_train_yclf.shape}")
print( f"splited data for model validating X : {model_val_X.shape},y : {model_val_y.shape}, y_clf : {model_val_yclf.shape}")
print( f"splited data for model testing X : {model_test_X.shape},y : {model_test_y.shape}, y_clf : {model_test_yclf.shape}")

Earliest date : 2024-01-01 00:00:00, Latest date 2024-12-31 00:00:00
       month
month       
1        717
2        709
3        828
4        845
5        823
6        822
7        882
8        857
9        831
10       933
11       794
12       795
splited data for model training X : (6483, 11),y : (6483,), y_clf : (6483,)
splited data for model validating X : (1764, 11),y : (1764,), y_clf : (1764,)
splited data for model testing X : (1589, 11),y : (1589,), y_clf : (1589,)


In [135]:
# Audit assertion on splited data
#validating the total splited data against original
total_split_X = len(model_train_X) + len(model_val_X) + len(model_test_X)
total_split_y = len(model_train_y) + len(model_val_y) + len(model_test_y)
total_split_yclf = len(model_train_yclf) + len(model_val_yclf) + len(model_test_yclf)
assert total_split_X == len(model_data_X), "total len of split data X must be equale to original"
assert total_split_y == len(model_data_y), "total len of split data y must be equale to original"
assert total_split_yclf == len(model_data_y_clf), "total len of split data yclf must be equale to original"
#ensure syncronization between  X and y and y_clf
train_sync = (model_train_X.index == model_train_y.index).all() and (model_train_X.index == model_train_yclf.index).all()
val_sync = (model_val_X.index == model_val_y.index).all() and (model_val_X.index == model_val_yclf.index).all()
test_sync = (model_test_X.index == model_test_y.index).all() and (model_test_X.index == model_test_yclf.index).all()
assert train_sync, "training features X and target y and yclf must be sync toghther"
assert val_sync, "validate features X and target y and yclf must be sync toghther"
assert test_sync, "test features X and target y and yclf must be sync toghther"
# assert that there is no date leakage within the splited dataset
assert model_train_X["fl_date"].max() < model_validate_X["fl_date"].min(), "leakage of dates between model_train and model_validate"
assert model_val_X["fl_date"].max() < model_test_X["fl_date"].min(), "leakage of dates between model_validate and model_test"

working on static analyses regarding the split data

In [10]:
# genral statics for train split data
print(model_train_X.shape)
print(f"train_y median {model_train_y.median()},train_y mean {model_train_y.mean()}, train_yclf median {model_train_yclf.median()},train_yclf mean {model_train_yclf.mean()}")
# genral statics for val split data
print(model_val_X.shape)
print(f"val_y median {model_val_y.median()},val_y mean {model_val_y.mean()}, val_yclf median {model_val_yclf.median()},val_yclf mean {model_val_yclf.mean()}")

# genral statics for test split data
print(model_test_X.shape)
print(f"test_y median {model_test_y.median()},test_y mean {model_test_y.mean()}, test_yclf median {model_test_yclf.median()},test_yclf mean {model_test_yclf.mean()}")

(6483, 11)
train_y median -5.0,train_y mean 10.328860095634736, train_yclf median 0.0,train_yclf mean 0.2390868425111831
(1764, 11)
val_y median -9.0,val_y mean -1.473922902494331, val_yclf median 0.0,val_yclf mean 0.1354875283446712
(1589, 11)
test_y median -7.0,test_y mean 6.199496538703587, test_yclf median 0.0,test_yclf mean 0.20767778477029578
